In [0]:
storage_account_name = "adlsretailmartabdul26"
storage_account_key = dbutils.secrets.get(scope="retailmart-kv-scope", key="adls-access-key")

# Configure Spark to use the account key for access
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# Test the connection by looking inside the bronze container
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"
display(dbutils.fs.ls(bronze_path))

path,name,size,modificationTime
abfss://bronze@adlsretailmartabdul26.dfs.core.windows.net/olist_customers_dataset.csv,olist_customers_dataset.csv,87490,1791381740000
abfss://bronze@adlsretailmartabdul26.dfs.core.windows.net/olist_order_items_dataset.csv,olist_order_items_dataset.csv,152143,1791381740000
abfss://bronze@adlsretailmartabdul26.dfs.core.windows.net/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,59309,1791381740000
abfss://bronze@adlsretailmartabdul26.dfs.core.windows.net/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,136909,1791381740000
abfss://bronze@adlsretailmartabdul26.dfs.core.windows.net/olist_orders_dataset.csv,olist_orders_dataset.csv,176373,1791381740000
abfss://bronze@adlsretailmartabdul26.dfs.core.windows.net/olist_products_dataset.csv,olist_products_dataset.csv,67211,1791381740000
abfss://bronze@adlsretailmartabdul26.dfs.core.windows.net/olist_sellers_dataset.csv,olist_sellers_dataset.csv,27506,1791381740000
abfss://bronze@adlsretailmartabdul26.dfs.core.windows.net/product_category_name_translation.csv,product_category_name_translation.csv,2612,1791381740000


In [0]:
from pyspark.sql.functions import col, to_timestamp

# Define paths
bronze_orders_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/olist_orders_dataset.csv"
silver_orders_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/orders"

# 1. Read raw CSV from Bronze
df_orders_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_orders_path)

# 2. Clean, cast timestamps, and filter null keys
df_orders_silver = df_orders_raw \
    .filter(col("order_id").isNotNull()) \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss"))

# 3. Write as Delta Lake into Silver
df_orders_silver.write.format("delta") \
    .mode("overwrite") \
    .save(silver_orders_path)

# 4. Preview the cleaned Silver data
display(spark.read.format("delta").load(silver_orders_path).limit(5))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
4e2a846769fc8af3af8b7826b058fe4f,a3b5428226ad4c042fed1da12cb6e683,delivered,2018-01-03T11:25:41Z,2018-01-03T11:34:23Z,2018-01-03T19:59:00Z,2018-01-11T20:34:05Z,2018-02-06T00:00:00Z
186ac860f8928db64590e1fdd744c9f5,c4fb1c069ba21ade36983cf6ad7069c8,shipped,2018-03-08T16:10:21Z,2018-03-08T16:56:03Z,2018-03-12T19:48:36Z,null,2018-03-28T00:00:00Z
f99c42cf20b64fe3dc045ae83e56f4e1,18af14116834f457513ba68d3c208eda,delivered,2017-07-11T21:23:14Z,2017-07-11T21:55:12Z,2017-07-14T13:52:18Z,2017-07-18T12:34:03Z,2017-07-24T00:00:00Z
8dca7901c45e1525ad654814a7db9eea,043e1a8a42664d4e6fd092d4b1f878e7,delivered,2018-01-05T01:35:24Z,2018-01-05T01:47:53Z,2018-01-05T22:58:11Z,2018-01-08T19:55:23Z,2018-01-23T00:00:00Z
981222fe88bbeab39c6ca1fa7bceba4c,2e8ff3002ea6077d8b4b45f771612d3a,delivered,2017-11-26T01:37:15Z,2017-11-26T01:50:04Z,2017-11-28T15:19:25Z,2017-12-04T20:59:23Z,2017-12-20T00:00:00Z


In [0]:
from pyspark.sql.types import FloatType

# Define paths
bronze_items_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/olist_order_items_dataset.csv"
silver_items_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/order_items"

# 1. Read raw CSV from Bronze
df_items_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_items_path)

# 2. Clean and cast numeric columns
df_items_silver = df_items_raw \
    .filter(col("order_id").isNotNull() & col("product_id").isNotNull()) \
    .withColumn("price", col("price").cast(FloatType())) \
    .withColumn("freight_value", col("freight_value").cast(FloatType()))

# 3. Write as Delta Lake into Silver
df_items_silver.write.format("delta") \
    .mode("overwrite") \
    .save(silver_items_path)

# 4. Preview the cleaned Silver data
display(spark.read.format("delta").load(silver_items_path).limit(5))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00143d0f86d6fbd9f9b38ab440ac16f5,1,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20T16:07:52Z,21.33,15.1
00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20T16:07:52Z,21.33,15.1
00143d0f86d6fbd9f9b38ab440ac16f5,3,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20T16:07:52Z,21.33,15.1
00a379dfab816a83741012b71b264098,1,786248c98cab9ad5cccdb751909da1af,1da3aeb70d7989d1e6d9b0e887f97c23,2018-08-07T20:04:20Z,15.99,7.39
01bc343311e5b8090b64d6ab287a5cf0,1,c8c969c3a6423f48df9fde8a13e6770b,b2479f944e1b90cf8a5de1bbfde284d6,2018-02-28T14:50:40Z,21.99,15.1
